In [7]:
%pip install yfinance azure-eventhub azure-identity

StatementMeta(, 5bfd24dc-3f86-43ba-bf87-850f55a794e8, 18, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [8]:
#df_stocks = spark.sql("SELECT StockSymbol, Max(TradeDate) FROM Stocks.dbo.dailystocks GROUP by StockSymbol")
df_stocks = spark.sql("""
    SELECT StockSymbol, MAX(TradeDate) AS LastTradeDate
    FROM Stocks.dbo.dailystocks
    WHERE Granularity = '5m'
    GROUP BY StockSymbol
""")
display(df_stocks)

StatementMeta(, 5bfd24dc-3f86-43ba-bf87-850f55a794e8, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0547307c-d6f2-4c53-8d97-c44b7083adc7)

In [9]:
import yfinance as yf
import pandas as pd
from datetime import datetime, date, timedelta
from pyspark.sql.functions import current_date, lit, col
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, LongType

StatementMeta(, 5bfd24dc-3f86-43ba-bf87-850f55a794e8, 21, Finished, Available, Finished, False)

In [10]:
def fetchStocks(stocks,startdate):

    today = datetime.today()

    df = yf.download(stocks, start=startdate, end=today, interval="5m")
    print(f"Downloaded {stocks} from {startdate} to {today} at 5m interval")

    df.columns = ['_'.join(col).strip() for col in df.columns.values]

    df_long = df.reset_index()
    df_long = df_long.rename(columns={df_long.columns[0]: 'Datetime'})

    df_long = pd.melt(
            df_long,           # Reset index to make Datetime a column
            id_vars=['Datetime'],       # Keep Datetime as is
            value_vars=[col for col in df.columns if col != 'Datetime'],  # All stock columns
            var_name='Temp',            # Temporary name for the column names
            value_name='Value'          # Temporary name for the values
        )

    df_long[['Metric', 'Symbol']] = df_long['Temp'].str.split('_',n=1, expand=True)

    df_long['Datetime'] = df_long['Datetime'].dt.tz_convert('UTC').dt.tz_localize(None)
    df_long = df_long.rename(columns={'Datetime': 'TradeDate'})
    df_long = df_long.rename(columns= {'Symbol': 'StockSymbol'} )

        # Pivot back so each metric becomes its own column
    df_5m = (df_long.pivot_table(
                index=['TradeDate', 'StockSymbol'],
                columns='Metric',
                values='Value')
                .reset_index()
                .rename_axis(None, axis=1))  # Remove the 'Metric' axis name
    df_5m['IngestionDate'] = datetime.today().strftime('%Y-%m-%d')
    df_5m['Dividends'] = 0
    df_5m['StockSplits'] = 0
    df_5m['Granularity'] = '5m'
    df_5m = df_5m[['StockSymbol','TradeDate', 'Open',  'High', 'Low', 'Close', 'Volume','Dividends','StockSplits','IngestionDate','Granularity']]


    print(df_5m)
    return df_5m
    

    schema = StructType([
    StructField("StockSymbol",   StringType(),    True),
    StructField("TradeDate",     TimestampType(), True),
    StructField("Open",          DoubleType(),    True),
    StructField("High",          DoubleType(),    True),
    StructField("Low",           DoubleType(),    True),
    StructField("Close",         DoubleType(),    True),
    StructField("Volume",        DoubleType(),    True),
    StructField("Dividends",     DoubleType(),    True),
    StructField("StockSplits",   DoubleType(),    True),
    StructField("IngestionDate", StringType(),    True),
    StructField("Granularity",   StringType(),    True),
    ])

    pyspark_df = spark.createDataFrame(df_5m, schema=schema)

    # ── Upsert instead of append ──────────────────────────────────────────────
    delta_table = DeltaTable.forName(spark, "DailyStocks")

    (
        delta_table.alias("target")
        .merge(
            pyspark_df.alias("source"),
            """
            target.StockSymbol = source.StockSymbol
            AND target.TradeDate  = source.TradeDate
            AND target.Granularity = source.Granularity
            """
        )
        .whenMatchedUpdate(set={
            "Open":          "source.Open",
            "High":          "source.High",
            "Low":           "source.Low",
            "Close":         "source.Close",
            "Volume":        "source.Volume",
            "Dividends":     "source.Dividends",
            "StockSplits":   "source.StockSplits",
            "IngestionDate": "source.IngestionDate",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    
    return True


  


StatementMeta(, 5bfd24dc-3f86-43ba-bf87-850f55a794e8, 22, Finished, Available, Finished, False)

In [11]:
all_results = []

for row in df_stocks.collect():
    
    stock = row['StockSymbol']
    last_date = row['LastTradeDate']
    
    today_dt = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0)

    if (today_dt - last_date).days >= 1:
        
        start_date = (last_date + timedelta(days=1)).strftime('%Y-%m-%d')
        print(f"Processing {stock} last ingested {last_date} starting {start_date} at 5m interval")
        result = fetchStocks(stock,start_date)
        
        if result is not None:
            all_results.append(result)

StatementMeta(, 5bfd24dc-3f86-43ba-bf87-850f55a794e8, 23, Finished, Available, Finished, False)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [13]:
from delta.tables import DeltaTable

if not all_results:
    print("Nothing to update — all stocks are current.")
else:
    df_all = pd.concat(all_results, axis=0, ignore_index=True)
    print(f"\nTotal rows to upsert: {len(df_all):,} across {df_all['StockSymbol'].nunique()} stocks")

    schema = StructType([
        StructField("StockSymbol",   StringType(),    True),
        StructField("TradeDate",     TimestampType(), True),
        StructField("Open",          DoubleType(),    True),
        StructField("High",          DoubleType(),    True),
        StructField("Low",           DoubleType(),    True),
        StructField("Close",         DoubleType(),    True),
        StructField("Volume",        DoubleType(),    True),
        StructField("Dividends",     DoubleType(),    True),
        StructField("StockSplits",   DoubleType(),    True),
        StructField("IngestionDate", StringType(),    True),
        StructField("Granularity",   StringType(),    True),
    ])

    pyspark_df = spark.createDataFrame(df_all, schema=schema)

    delta_table = DeltaTable.forName(spark, "DailyStocks")

    (
        delta_table.alias("target")
        .merge(
            pyspark_df.alias("source"),
            """
            target.StockSymbol  = source.StockSymbol
            AND target.TradeDate   = source.TradeDate
            AND target.Granularity = source.Granularity
            """
        )
        .whenMatchedUpdate(set={
            "Open":          "source.Open",
            "High":          "source.High",
            "Low":           "source.Low",
            "Close":         "source.Close",
            "Volume":        "source.Volume",
            "Dividends":     "source.Dividends",
            "StockSplits":   "source.StockSplits",
            "IngestionDate": "source.IngestionDate",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

    final_count = spark.table("DailyStocks").count()
    print(f"\n✅ Done. 'DailyStocks' now contains {final_count:,} rows.")

StatementMeta(, 5bfd24dc-3f86-43ba-bf87-850f55a794e8, 25, Finished, Available, Finished, False)


Total rows to upsert: 1,375 across 18 stocks

✅ Done. 'DailyStocks' now contains 150,002 rows.
